Ok, so I need to make an HR diagram. I should manually subdivide the target_table by spectral type inside this notebook instead of creating a bunch of separate files externally, or making the script itself do the separate plotting.

In [1]:

from __future__ import print_function


import matplotlib
matplotlib.use('pdf')



import numpy as np
#from astroquery.gaia import Gaia
import astropy.units as u
import os
import sys
import astropy.coordinates as coord
from astropy.table import Table, QTable
import matplotlib.pyplot as plt
import scipy.stats as scistats
#import seaborn as sns
#import astropy

import time
start = time.time()


I do need to figure out how to get the Gaia scripts into my import path though because I'm not currently over there.... Maybe I should just move this over into the Gaia directory, externally? That's probably the easiest solution to this problem. I'm going to just move this jupyter notebook.

Ok, I have moved this into the Gaia directory and restarted the kernel. Now let's see if this import works...

In [2]:

#import passband_model_convolution as pmc
import gaia_extinction
#import wdatmos
import plotting_dicts as pod

And import the actual script that does the plotting

In [3]:
import plot_alt_cmd as pac

Distance-limited Sample like Figure 6 from DR2HRD


Also need to import the panstarrs plotting code for later use

In [4]:
import plot_panstarrs2 as pps2



Cleaning table.
Starting from 146
Cleaning  g
New count: 146
Cleaning  r
New count: 146
Cleaning  i
New count: 146
Cleaning  z
New count: 146
Cleaning  y
New count: 146


So, it did execute all of that crap like I expected, and it most likely assigned a bunch of variables that I'll now need to be careful of... actually those should have been assigned inside the script I imported, so it should be pac.variable, so I should be safe to do stuff

So the problem is the import never stops running. I added in the "if" statement that makes the plotting stuff only execute if the script is the main file called, so that should fix it. Now I'll restart the damn kernel again...

In [5]:
#target_input ='20190516B_retargeted_purple_search_gaia_scbd.csv'
#target_input ='20190516B_retargeted_purple_search_gaia_scbd_20201102_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20210616_update.csv'
#target_input ='20190516B_retargeted_purple_search_gaia_scbd_20210109_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20210824_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20211117_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20220106_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20220106_update_ps2_phot_added.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20220503_update_ps1_phot_added.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20220804_update.csv'
#target_input='20190516B_retargeted_purple_search_gaia_scbd_20230131_update.csv'
target_input='20190516B_retargeted_purple_search_gaia_scbd_20230301_update.csv'
figure_output_dir='/Users/BenKaiser/Desktop/'


In [6]:
target_table = Table.read(target_input)


Ok, I need to make this generate a nice litte BP-RP H-R diagram with Nicola's cuts drawn onto it.

In [7]:
plt.rc('font',family='Microsoft Sans Serif',size=12)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
#plt.figure(figsize=(9.5,9.67), constrained_layout=True)
plt.figure(figsize=(6.5,9.67/9.5*6.5), constrained_layout=False)


pac.plot_bkg_cmd(colours=['bp','rp'])
pac.plot_nicola_cuts()
pac.plot_nicola_flag()
plt.ylabel(r'$G_{{abs}}$')
plt.xlabel(r'$G_{BP} - G_{RP}$')
plt.ylim(17,-2.5)
plt.xlim(-0.75,5)
plt.subplots_adjust(wspace = 0, hspace = 0, top = 0.90, bottom = 0.10, left = 0.12, right = 0.90)
plt.legend()
print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig("HR_diagram_"+time_string+'.pdf')#plt.grid(True)

plt.show()


#plt.show()

(22288,)
/Users/BenKaiser/Desktop/gaia
/Users/BenKaiser/Desktop
1677716323.8609338


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:25: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


Ay oh! It worked! Now onto the division by group. I need to check what the column name is... And I'll probably need to reimpose it onto the CSV again...

In [8]:
#sub_vals= np.where(target_table['good_obs']== 1)
#sub_vals=np.where(target_table['gf19']=='TRUE')

In [9]:
#sub_table= target_table[sub_vals]
#I no longer want some spinoff table that I made for no apparent reason
sub_table=target_table

In [10]:
dz= np.where(sub_table['sp_type']=='DZNa')

In [11]:
print(sub_table[dz])
print('rows=',len(sub_table[dz]))
num_dz=len(sub_table[dz])

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
8.01e-06    1.64e+18 b'Gaia DR2 4353607450860305024' ...           1        786
4.76e-06    1.64e+18 b'Gaia DR2 2341622358827194880' ...           1        819
rows= 2


In [12]:
wddm= np.where(sub_table['sp_type']=='WDdM')

In [13]:
print(sub_table[wddm])
print('rows=',len(sub_table[wddm]))
num_wddm=len(sub_table[wddm])

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
1.63e-06    1.64e+18 b'Gaia DR2 4913565903725184640' ...          --        703
3.28e-07    1.64e+18 b'Gaia DR2 4930805868092016256' ...          --        704
7.95e-06    1.64e+18 b'Gaia DR2 2593774421282488832' ...           3        705
8.04e-06    1.64e+18   b'Gaia DR2 37101100029189376' ...           3        719
6.91e-06    1.64e+18 b'Gaia DR2 3192967924382400256' ...           3        721
5.98e-06    1.64e+18 b'Gaia DR2 5091988397208440448' ...           3        722
8.07e-06    1.64e+18 b'Gaia DR2 3000597125173673088' ...           3        739
7.28e-06    1.64e+18 b'Gaia DR2 5559594371631444736' ...          --        741
6.52e-06    1.64e+18 b'Gaia DR2 5270620145795842432' ...          --        744
7.85e-06    1.64e+18  b'Gaia DR2 683016588814929280' ...           3        749
4.94e-06    1.64e+18 b'Gaia DR2 39927224

In [14]:
dc= np.where(sub_table['sp_type']=='DC')

In [15]:
print(sub_table[dc])
print('rows=',len(sub_table[dc]))
num_dc=len(sub_table[dc])

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
4.21e-06    1.64e+18 b'Gaia DR2 2336521827465439616' ...           4        700
8.26e-06    1.64e+18 b'Gaia DR2 2802321579156605568' ...           4        701
 2.1e-06    1.64e+18 b'Gaia DR2 4909222871450620416' ...          --        702
2.55e-06    1.64e+18 b'Gaia DR2 4712157368044092928' ...          --        706
7.24e-06    1.64e+18 b'Gaia DR2 2521858084423378176' ...           4        708
2.41e-06    1.64e+18 b'Gaia DR2 4967454759604815360' ...          --        709
4.88e-06    1.64e+18 b'Gaia DR2 5156753037993200512' ...           4        711
8.27e-06    1.64e+18  b'Gaia DR2 109336535778050944' ...           4        713
7.88e-06    1.64e+18   b'Gaia DR2 17497872857713920' ...           4        715
6.15e-06    1.64e+18 b'Gaia DR2 5115934390366550656' ...           4        718
     ...         ...                    

In [16]:
dqpec=np.where(sub_table['sp_type']=='DQpec')

In [17]:
print(sub_table[dqpec])
print('rows=',len(sub_table[dqpec]))
num_dqpec=len(sub_table[dqpec])

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
4.12e-06    1.64e+18 b'Gaia DR2 5140433498003756160' ...           4        707
3.54e-06    1.64e+18 b'Gaia DR2 4673772519470806144' ...          --        717
8.79e-06    1.64e+18 b'Gaia DR2 1761085712628042112' ...           2        805
5.03e-06    1.64e+18 b'Gaia DR2 6596688068617463808' ...          --        811
rows= 4


In [18]:
unknown=np.where(sub_table['sp_type']=='??')
print(sub_table[unknown])
print('rows=',len(sub_table[unknown]))
num_unknown=len(sub_table[unknown])

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
2.88e-06    1.64e+18 b'Gaia DR2 5051990775531173120' ...          --        712
   3e-06    1.64e+18 b'Gaia DR2 4849260661232437504' ...          --        716
 5.6e-06    1.64e+18 b'Gaia DR2 4768125052520236032' ...          --        733
7.78e-06    1.64e+18 b'Gaia DR2 2943040474599473024' ...           5        737
7.08e-06    1.64e+18 b'Gaia DR2 5581206101598276992' ...          --        740
8.46e-06    1.64e+18 b'Gaia DR2 6187146135032473344' ...           4        763
7.95e-06    1.64e+18 b'Gaia DR2 6891929583743053696' ...           2        807
5.51e-06    1.64e+18 b'Gaia DR2 6368907055055475968' ...          --        809
rows= 8


In [19]:
len(sub_table)

120

In [20]:
sub_table.pprint()

  dist   solution_id           designation           ... sed_sp_type target_num
-------- ----------- ------------------------------- ... ----------- ----------
4.21e-06    1.64e+18 b'Gaia DR2 2336521827465439616' ...           4        700
8.26e-06    1.64e+18 b'Gaia DR2 2802321579156605568' ...           4        701
 2.1e-06    1.64e+18 b'Gaia DR2 4909222871450620416' ...          --        702
1.63e-06    1.64e+18 b'Gaia DR2 4913565903725184640' ...          --        703
3.28e-07    1.64e+18 b'Gaia DR2 4930805868092016256' ...          --        704
7.95e-06    1.64e+18 b'Gaia DR2 2593774421282488832' ...           3        705
2.55e-06    1.64e+18 b'Gaia DR2 4712157368044092928' ...          --        706
4.12e-06    1.64e+18 b'Gaia DR2 5140433498003756160' ...           4        707
7.24e-06    1.64e+18 b'Gaia DR2 2521858084423378176' ...           4        708
2.41e-06    1.64e+18 b'Gaia DR2 4967454759604815360' ...          --        709
     ...         ...                    

In [21]:
#for row in sub_table:
#    print(row['sp_type'])

In [22]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,5))
#colours= ['g', 'rp']
colours= ['bp', 'rp']

In [23]:
pac.plot_bkg_cmd(colours=colours)

(22288,)


In [24]:
pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey')

g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 3.9942887752886236e-07
rp_calc - rp_measured 3.959468415359879e-07
1.4843978934820363 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 2.9424750636053432e-08
rp_calc - rp_measured 7.789809970404349e-07
1.4597580104437569 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.893551400821707e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.2465963645092906 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 2.1880904199633733e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 8.792550758585094e-07
rp_calc - rp_measured -8.149073202901036e-07
1.0707204641623953 14.554334895776861
GaiaJ0135-6306
g_calc-g_measured -7.849035981166708e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured

bp_calc-bp_measured -6.919502517632736e-07
rp_calc - rp_measured -3.7216843296050683e-07
1.5534970002181794 15.691956853506909
GaiaJ1511+2728
g_calc-g_measured -8.086345815172535e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 5.045467510456092e-07
rp_calc - rp_measured 8.801175823691665e-07
0.9843898844291665 14.955752587293583
SDSSJ1524+2010
g_calc-g_measured -5.113692580493989e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -1.5524582153148003e-07
rp_calc - rp_measured -5.603957085043021e-07
1.4546303051498874 14.51845252823491
GaiaJ1534-0935
g_calc-g_measured -4.815700975768777e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -4.377681150913304e-08
rp_calc - rp_measured 7.395510337460109e-07
1.280381366672156 16.01067716334324
SDSSJ1536+1718
g_calc-g_measured -1.0460516719490442e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -2.1913393766226363e-08
rp_calc - rp_measured -4.33651898390508e-07
1.6950725917385

In [25]:
pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta')

g_calc-g_measured -6.927051501293136e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 7.885571591259577e-07
rp_calc - rp_measured 2.0154264745997352e-07
1.7183519570145123 15.983156795809741
GaiaJ1644-0449
g_calc-g_measured 9.295174656642757e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -3.4206359700306166e-07
rp_calc - rp_measured 6.894105979426968e-07
1.8052624885258055 15.933263242899619
WDJ2356-209


In [26]:
pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r')

g_calc-g_measured 6.192432451257446e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 3.568661384178995e-07
rp_calc - rp_measured -6.388349191865927e-07
1.3187456557010577 14.654582033860546
GaiaJ0110-5600
g_calc-g_measured -1.9354945735017282e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -8.529129686962733e-07
rp_calc - rp_measured -2.1320193610563365e-08
2.577529078407224 14.398589010194087
GaiaJ0126-4707
g_calc-g_measured 4.32181977316759e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -5.959133702049257e-07
rp_calc - rp_measured -8.246310656545575e-08
1.7205957965497376 14.650036588087136
SDSSJ0132+1823
g_calc-g_measured 3.7945259023786093e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.415381044178048e-07
rp_calc - rp_measured -6.967919858880123e-07
2.1110853183300904 15.181730070341192
GaiaJ0349+1241
g_calc-g_measured 5.400433664703996e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -

In [27]:
pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2')

g_calc-g_measured -8.019691932759088e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.458464874015135e-07
rp_calc - rp_measured 2.5982915019540087e-07
1.5229095460173383 15.685338645993514
LEHPM1774
g_calc-g_measured -5.881854967526579e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -5.44525363466164e-07
rp_calc - rp_measured 4.959636719092941e-07
1.8254021695109621 15.140734424709395
GaiaJ0334-6348
g_calc-g_measured -3.387537752530534e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 7.825520249582496e-07
rp_calc - rp_measured -3.164725725923745e-07
1.9622870890245956 15.39159628926175
GaiaJ2053+1302
g_calc-g_measured -4.6784677820710385e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 4.474943544607868e-07
rp_calc - rp_measured -6.432318251370361e-07
1.2901202607261801 15.016479459303516
GaiaJ2236-3635


In [28]:
#plt.xlabel(r'$G-G_{RP}$')
plt.xlabel(r'$G_{BP}-G_{RP}$')
plt.ylabel(r'$M_{G}$')
#plt.xlim(-0.5, 2)

Text(0, 0.5, '$M_{G}$')

In [29]:
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:1: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  """Entry point for launching an IPython kernel.


## BP-RP H-R diagram

In [30]:
#plt.rc('font',size=12)
plt.rc('font',family='Microsoft Sans Serif',size=12)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
plt.figure(figsize=(9.5,9.67), constrained_layout=False)

#colours= ['g', 'rp']
colours= ['bp', 'rp']
pac.plot_bkg_cmd(colours=colours)

pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey')
pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta')
pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r')
pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2')
pac.plot_nicola_cuts()

#plt.xlabel(r'$G-G_{RP}$')

plt.xlabel(r'$G_{BP}-G_{RP}$')
plt.ylabel(r'$M_{G}$')
#plt.legend()

#plt.xlim(0.5, 1.5)
#plt.ylim(17.5,12.5)

plt.ylim(17,-1.5)
plt.xlim(-0.75,5)


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig("HR_diagram_"+time_string+'.pdf')#plt.grid(True)

plt.show()




(22288,)
g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 3.9942887752886236e-07
rp_calc - rp_measured 3.959468415359879e-07
1.4843978934820363 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 2.9424750636053432e-08
rp_calc - rp_measured 7.789809970404349e-07
1.4597580104437569 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.893551400821707e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.2465963645092906 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 2.1880904199633733e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 8.792550758585094e-07
rp_calc - rp_measured -8.149073202901036e-07
1.0707204641623953 14.554334895776861
GaiaJ0135-6306
g_calc-g_measured -7.849035981166708e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp

g_calc-g_measured 6.192432451257446e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 3.568661384178995e-07
rp_calc - rp_measured -6.388349191865927e-07
1.3187456557010577 14.654582033860546
GaiaJ0110-5600
g_calc-g_measured -1.9354945735017282e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -8.529129686962733e-07
rp_calc - rp_measured -2.1320193610563365e-08
2.577529078407224 14.398589010194087
GaiaJ0126-4707
g_calc-g_measured 4.32181977316759e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -5.959133702049257e-07
rp_calc - rp_measured -8.246310656545575e-08
1.7205957965497376 14.650036588087136
SDSSJ0132+1823
g_calc-g_measured 3.7945259023786093e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.415381044178048e-07
rp_calc - rp_measured -6.967919858880123e-07
2.1110853183300904 15.181730070341192
GaiaJ0349+1241
g_calc-g_measured 5.400433664703996e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:38: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [31]:
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:1: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  """Entry point for launching an IPython kernel.


In [32]:
parallax = sub_table['parallax']+pac.parallax_correction
parallax = parallax*1e-3
distance = 1./parallax
absmags=[]
#for row in sub_table:
    #this_abs, throw, garbage= pac.get_pass_abs_mag(target_table, plot_all=False, passband_string='g')
    #absmags.append(this_abs)

In [33]:
print(len(sub_table[dz]))
print(len(sub_table[dqpec]))
print(len(sub_table[dc]))
print(len(sub_table[wddm]))
print(len(sub_table))

2
4
53
16
120


## G-RP H-R diagram

In [34]:
#plt.rc('font',size=12)
plt.rc('font',family='Microsoft Sans Serif',size=12)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
plt.figure(figsize=(6.5,9.67/9.5*6.5), constrained_layout=False)

colours= ['g', 'rp']
#colours= ['bp', 'rp']
pac.plot_bkg_cmd(colours=colours)
pac.plot_target_table(target_table, colours=colours, list_color='#AB5DEE',markersize=2,annotate=False)
#pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey')
#pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r')
#pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2')
#pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta')
#pac.plot_nicola_cuts()
pac.plot_ben_cuts()

plt.xlabel(r'$G-G_{RP}$')

#plt.xlabel(r'$G_{BP}-G_{RP}$')
#plt.ylabel(r'$M_{G}$')
plt.ylabel(r'$G_{abs}$')


#plt.legend()

#plt.xlim(0.5, 1.5)
#plt.ylim(17.5,12.5)

plt.ylim(17,-1.5)
plt.xlim(-0.4,2.0)


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig("HR_diagram_GRP_"+time_string+'.pdf')#plt.grid(True)

plt.show()





(22981,)
g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 3.1008049461433984e-08
rp_calc - rp_measured 3.959468415359879e-07
1.0138221850612084 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 1.3516314112393957e-07
rp_calc - rp_measured 7.789809970404349e-07
0.9510014761821459 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 4.3796919158012315e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.0376613331233422 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 6.192432451257446e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 6.192432451257446e-07
rp_calc - rp_measured -6.388349191865927e-07
0.9681389980781638 14.654582033860546
GaiaJ0110-5600
g_calc-g_measured -1.9354945735017282e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_mea

1.2009849833711854 15.236859277762573
WISEA0615-1247
g_calc-g_measured 6.678550334981992e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 6.678550334981992e-08
rp_calc - rp_measured -4.641951285577761e-07
1.0519280909806312 15.645021866347417
GaiaJ0628-3602
g_calc-g_measured 7.680212803506947e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 7.680212803506947e-07
rp_calc - rp_measured -6.31612245172164e-07
1.0522055296335253 15.125934745507424
GaiaJ0706-4246
g_calc-g_measured -2.7634125032705015e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -2.7634125032705015e-07
rp_calc - rp_measured -6.589159333714179e-07
0.978165062574682 14.742383822950153
GaiaJ0736-4256
g_calc-g_measured 6.434582786596366e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 6.434582786596366e-07
rp_calc - rp_measured 5.678956540577929e-07
1.020370555562625 15.82257523069756
GaiaJ0738-4339
g_calc-g_measured -7.961360957153829e-07
index_length 10000


g_calc-bp_measured -6.927051501293136e-07
rp_calc - rp_measured 2.0154264745997352e-07
1.1300535857522007 15.983156795809741
GaiaJ1644-0449
g_calc-g_measured 5.05970053410465e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 5.05970053410465e-07
rp_calc - rp_measured -3.883558363781958e-07
1.2549008743258874 14.629662112975334
GaiaJ1700+2713
g_calc-g_measured -6.482064129897935e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -6.482064129897935e-07
rp_calc - rp_measured 7.809636812794452e-07
0.8852315008299065 14.624830268902137
--
g_calc-g_measured 7.488105602249107e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 7.488105602249107e-07
rp_calc - rp_measured -8.104940718567377e-08
1.049378269859968 16.01173635815666
--
g_calc-g_measured -6.03358426332079e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -6.03358426332079e-07
rp_calc - rp_measured 3.864620623517112e-07
0.9904088601795102 15.541587513808384
--
g_calc-g_mea

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:42: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


## Zoomed in G-RP H-R diagram

In [35]:
#plt.rc('font',size=12)
plt.rc('font',family='Microsoft Sans Serif',size=12)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
#plt.figure(figsize=(9.5,6), constrained_layout=False)
plt.figure(figsize=(6.5,6/9.5*6.5), constrained_layout=True)


colours= ['g', 'rp']
#colours= ['bp', 'rp']
pac.plot_bkg_cmd(colours=colours)
#pac.plot_target_table(target_table, colours=colours, list_color='#AB5DEE',markersize=6,annotate=False)
pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey',label='DC-'+str(num_dc))
pac.plot_target_table(sub_table[unknown], colours=colours, list_color='#006652',label='Unknown-'+str(num_unknown))
pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta',label='DZ-'+str(num_dz))
pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r',label='WD+dM-'+str(num_wddm))
pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2',label='DQpec-'+str(num_dqpec))

#pac.plot_nicola_cuts()
pac.plot_ben_cuts()

plt.xlabel(r'$G-G_{RP}$')

#plt.xlabel(r'$G_{BP}-G_{RP}$')
#plt.ylabel(r'$M_{G}$')
plt.ylabel(r'$G_{abs}$')


#plt.legend()

plt.xlim(0.4, 1.8)
plt.ylim(17.5,13.5)

#plt.ylim(17,-1.5)
#plt.xlim(-0.4,2.0)
plt.legend(loc='lower left',fontsize=11)

print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig("HR_diagram_GRP_zoomed"+time_string+'.pdf')#plt.grid(True)

plt.show()






(22981,)
g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 3.1008049461433984e-08
rp_calc - rp_measured 3.959468415359879e-07
1.0138221850612084 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 1.3516314112393957e-07
rp_calc - rp_measured 7.789809970404349e-07
0.9510014761821459 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 4.3796919158012315e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.0376613331233422 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 2.1880904199633733e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 2.1880904199633733e-08
rp_calc - rp_measured -8.149073202901036e-07
0.9801911167882231 14.554334895776861
GaiaJ0135-6306
g_calc-g_measured -7.849035981166708e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_me

g_calc-bp_measured -5.113692580493989e-07
rp_calc - rp_measured -5.603957085043021e-07
0.8711071490264501 14.51845252823491
GaiaJ1534-0935
g_calc-g_measured -4.815700975768777e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -4.815700975768777e-07
rp_calc - rp_measured 7.395510337460109e-07
1.006828988878869 16.01067716334324
SDSSJ1536+1718
g_calc-g_measured -1.0460516719490442e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -1.0460516719490442e-07
rp_calc - rp_measured -4.33651898390508e-07
1.153124189046732 15.519061544618307
GaiaJ1801+1806
g_calc-g_measured -6.844901818681137e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -6.844901818681137e-07
rp_calc - rp_measured -7.802888823960075e-07
0.9843464857986994 15.005388496574362
GaiaJ1822+2319
g_calc-g_measured 3.73787141683124e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 3.73787141683124e-07
rp_calc - rp_measured -8.473740749082026e-07
1.0561020611612157 14.551

g_calc-g_measured -3.387537752530534e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -3.387537752530534e-07
rp_calc - rp_measured -3.164725725923745e-07
1.0426807177187953 15.39159628926175
GaiaJ2053+1302
g_calc-g_measured -4.6784677820710385e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -4.6784677820710385e-07
rp_calc - rp_measured -6.432318251370361e-07
0.9546109953850461 15.016479459303516
GaiaJ2236-3635
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1677716338.7298088


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:46: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [36]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,5))
#colours= ['bp', 'rp']
colours= ['g', 'rp']
pac.plot_bkg_cmd(colours=colours)
#pac.plot_nicola_cuts()
pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey', label='DC')
pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r', label='WD+dM')
pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2', label='DQpec')
pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta', label='DZ')
plt.xlabel(r'$G-G_{RP}$')
#plt.xlabel(r'$G_{BP}-G_{RP}$')
plt.ylabel(r'$M_{G}$')

#plt.xlim(-0.5, 1.5)
#plt.ylim(17.5,7.5)

plt.xlim(-0.4, 2.0)
#plt.xlim(0,4)
plt.ylim(17.5,10)



plt.legend()
plt.legend(loc='lower left')



(22981,)
g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 3.1008049461433984e-08
rp_calc - rp_measured 3.959468415359879e-07
1.0138221850612084 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 1.3516314112393957e-07
rp_calc - rp_measured 7.789809970404349e-07
0.9510014761821459 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 4.3796919158012315e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.0376613331233422 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 2.1880904199633733e-08
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 2.1880904199633733e-08
rp_calc - rp_measured -8.149073202901036e-07
0.9801911167882231 14.554334895776861
GaiaJ0135-6306
g_calc-g_measured -7.849035981166708e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_me

g_calc-bp_measured 3.714720264724747e-07
rp_calc - rp_measured -3.8756014220098223e-07
1.0701244290321696 15.702157820608221
SDSSJ2314+2717
g_calc-g_measured 7.736767528854216e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 7.736767528854216e-07
rp_calc - rp_measured 5.452833669039592e-07
0.8555071283933842 14.222606859108277
GaiaJ2320-6637
g_calc-g_measured -9.552558672965006e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -9.552558672965006e-07
rp_calc - rp_measured -8.304623939636713e-07
0.8754004252065286 14.505917391040104
GaiaJ2324-5942
g_calc-g_measured 6.192432451257446e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured 6.192432451257446e-07
rp_calc - rp_measured -6.388349191865927e-07
0.9681389980781638 14.654582033860546
GaiaJ0110-5600
g_calc-g_measured -1.9354945735017282e-07
index_length 10000
mag_dist.shape (10000,)
g_calc-bp_measured -1.9354945735017282e-07
rp_calc - rp_measured -2.1320193610563365e-08
1.3014429277707364 

In [37]:
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:1: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  """Entry point for launching an IPython kernel.


In [38]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,5))
colours= ['bp', 'rp']
#colours= ['g', 'rp']
pac.plot_bkg_cmd(colours=colours)
pac.plot_nicola_cuts()
pac.plot_target_table(sub_table[dc], colours=colours, list_color='grey', label='DC')
pac.plot_target_table(sub_table[wddm],colours=colours, list_color='r', label='WD+dM')
pac.plot_target_table(sub_table[dqpec],colours=colours, list_color=  '#1ca1f2', label='DQpec')
pac.plot_target_table(sub_table[dz], colours=colours, list_color= 'magenta', label='DZ')
#plt.xlabel(r'$G-G_{RP}$')
plt.xlabel(r'$G_{BP}-G_{RP}$')
plt.ylabel(r'$M_{G}$')

#plt.xlim(-0.5, 1.5)
#plt.ylim(17.5,7.5)

#plt.xlim(-0.25, 1.5)
plt.xlim(0,4)
plt.ylim(17.5,10)



plt.legend()
#plt.legend(loc='lower left')
plt.legend(loc='upper right')






(22288,)
g_calc-g_measured 3.1008049461433984e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 3.9942887752886236e-07
rp_calc - rp_measured 3.959468415359879e-07
1.4843978934820363 15.836777863860455
GaiaJ0008-2404
g_calc-g_measured 1.3516314112393957e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 2.9424750636053432e-08
rp_calc - rp_measured 7.789809970404349e-07
1.4597580104437569 15.080193003839463
SDSSJ0049+2244
g_calc-g_measured 4.3796919158012315e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.893551400821707e-07
rp_calc - rp_measured -2.9515415178593685e-07
1.2465963645092906 15.461686437323877
GaiaJ0110-5926
g_calc-g_measured 2.1880904199633733e-08
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 8.792550758585094e-07
rp_calc - rp_measured -8.149073202901036e-07
1.0707204641623953 14.554334895776861
GaiaJ0135-6306
g_calc-g_measured -7.849035981166708e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp

bp_calc-bp_measured 3.568661384178995e-07
rp_calc - rp_measured -6.388349191865927e-07
1.3187456557010577 14.654582033860546
GaiaJ0110-5600
g_calc-g_measured -1.9354945735017282e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -8.529129686962733e-07
rp_calc - rp_measured -2.1320193610563365e-08
2.577529078407224 14.398589010194087
GaiaJ0126-4707
g_calc-g_measured 4.32181977316759e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -5.959133702049257e-07
rp_calc - rp_measured -8.246310656545575e-08
1.7205957965497376 14.650036588087136
SDSSJ0132+1823
g_calc-g_measured 3.7945259023786093e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured 6.415381044178048e-07
rp_calc - rp_measured -6.967919858880123e-07
2.1110853183300904 15.181730070341192
GaiaJ0349+1241
g_calc-g_measured 5.400433664703996e-07
index_length 10000
mag_dist.shape (10000,)
bp_calc-bp_measured -2.6335691316603516e-07
rp_calc - rp_measured -2.9314737304275695e-07
2.147903469790

In [39]:
label_pos=3.3
label_off=20
x_pos= 4500
y_pos= 0.92

2021-06-16 updated version counts:

2

4

26

12

120

## PanSTARRS-2 Color-Color diagrams (2022-05-02)

import the panstarrs plotting code I wrote externally earlier

In [40]:
plt.show()

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:1: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  """Entry point for launching an IPython kernel.


In [41]:
#plt.rc('font',size=12)
plt.rc('font',family='Microsoft Sans Serif',size=18)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
plt.figure(figsize=(9.5,6), constrained_layout=False)

#c1c2='g-r'
#c3c4='r-i'

c1c2='g-y'
c3c4='r-i'



pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table,color=  'k',label='Obj. w/out spectrum')
pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dc],color='grey',label='DC-'+str(num_dc))
pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[unknown], color='#006652',label='??-'+str(num_unknown))
pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dz], color= 'magenta',label='DZ-'+str(num_dz))
pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[wddm],color='r',label='WD+dM-'+str(num_wddm))
pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dqpec],color=  '#1ca1f2',label='DQpec-'+str(num_dqpec))





#pac.plot_nicola_cuts()
#pac.plot_ben_cuts()

#plt.xlabel(r'$G-G_{RP}$')

#plt.xlabel(r'$G_{BP}-G_{RP}$')
#plt.ylabel(r'$M_{G}$')
#plt.legend()

#plt.xlim(0.4, 1.8)
#plt.ylim(17.5,13.5)

#plt.ylim(17,-1.5)
#plt.xlim(-0.4,2.0)
plt.legend(loc='best',fontsize=16)

print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig(c3c4+'_vs_'+c1c2+'_'+time_string+'.pdf')#plt.grid(True)

plt.show()



KeyError: 'ps1_g_mean_mag'

#plt.rc('font',size=12)
plt.rc('font',family='Microsoft Sans Serif',size=18)
#fig= plt.figure(figsize=(10,5))
#plt.figure(figsize=(7.25,9.67), constrained_layout=False)
plt.figure(figsize=(9.5,6), constrained_layout=False)

c1c2='g-r'
c3c4='r-i'

color_list=['g','r','i','z','y']

for color1 in color_list:
    for color2 in color_list:
        c1c2=color1+'-'+color2
        print(c1c2)
        if color1 != color2:
            for color3 in color_list:
                for color4 in color_list:
                    c3c4=color3+'-'+color4
                    print(c3c4)
                    if ((color3 != color4) and (c1c2 != c3c4)):
                        #do the plotting
                        print('should be plotting')

                        pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dc],color='grey',label='DC-'+str(num_dc))
                        pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[unknown], color='#006652',label='??-'+str(num_unknown))
                        pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dz], color= 'magenta',label='DZ-'+str(num_dz))
                        pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[wddm],color='r',label='WD+dM-'+str(num_wddm))
                        pps2.plot_colors(c1c2=c1c2, c3c4=c3c4,input_table=sub_table[dqpec],color=  '#1ca1f2',label='DQpec-'+str(num_dqpec))




                        #pac.plot_nicola_cuts()
                        #pac.plot_ben_cuts()

                        #plt.xlabel(r'$G-G_{RP}$')

                        #plt.xlabel(r'$G_{BP}-G_{RP}$')
                        #plt.ylabel(r'$M_{G}$')
                        #plt.legend()

                        #plt.xlim(0.4, 1.8)
                        #plt.ylim(17.5,13.5)

                        #plt.ylim(17,-1.5)
                        #plt.xlim(-0.4,2.0)
                        plt.legend(loc='best',fontsize=16)

                        print(os.getcwd())
                        os.chdir(figure_output_dir)
                        print(os.getcwd())
                        start = time.time()
                        print(start)
                        time_string=str(start).split('.')[0]
                        plt.savefig(c3c4+'_vs_'+c1c2+'_'+time_string+'.pdf')#plt.grid(True)

                        plt.show()


                    else:
                        pass
        else:
            pass

In [ ]:
(2+50+16+5)/83.